# 02 Classic models

This notebook trains and compares classical TF-IDF models for multi-label classification and regression.

In [7]:
from pathlib import Path
import sys

_PROJECT_MARKERS = (
    Path('src') / 'data' / 'prepare_dataset.py',
    Path('Turismy') / 'reviews.csv',
)


def _is_project_root(path):
    return all((path / marker).exists() for marker in _PROJECT_MARKERS)


def _candidate_roots(start):
    start = start.resolve()
    seen = set()

    for candidate in (start, *start.parents):
        if candidate not in seen:
            seen.add(candidate)
            yield candidate

    for base in (start, start.parent):
        try:
            children = list(base.iterdir())
        except OSError:
            continue

        for child in children:
            try:
                if not child.is_dir():
                    continue
                candidate = child.resolve()
            except OSError:
                continue

            if candidate not in seen:
                seen.add(candidate)
                yield candidate


def find_project_root():
    for candidate in _candidate_roots(Path.cwd()):
        if _is_project_root(candidate):
            return candidate

    raise RuntimeError(
        'Project root was not found. Start the notebook from the repository root, '
        'the notebooks folder, or the parent folder that contains MasinskoUcenje-Projekat.'
    )


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
import json

import pandas as pd
from IPython.display import display

from src.data.prepare_dataset import prepare_dataset
from src.models.train_classic import train_classification_models, save_classification_result
from src.models.train_regression import train_regression_models, save_regression_result

In [8]:
dataset_path = PROJECT_ROOT / 'Turismy' / 'reviews_enriched.csv'
if not dataset_path.exists():
    df = prepare_dataset(output_path=dataset_path)
else:
    df = pd.read_csv(dataset_path)

df.shape

(9964, 8)

In [10]:
classification_result = train_classification_models(df)
save_classification_result(classification_result)

classification_table = (
    pd.DataFrame(classification_result.all_metrics)
    .T
    .drop(columns=['per_label'])
    .sort_values('micro_f1', ascending=False)
)
classification_per_label_table = pd.concat(
    {
        model_name: pd.DataFrame(metrics['per_label']).T
        for model_name, metrics in classification_result.all_metrics.items()
    },
    names=['model', 'label'],
)

print(f'Najbolji model: {classification_result.name}')
print('Ukupne metrike za sve classification modele:')
display(classification_table)
print('Metrike po labelama za svaki classification model:')
classification_per_label_table


Najbolji model: linear_svm
Ukupne metrike za sve classification modele:


,micro_precision,micro_recall,micro_f1,macro_precision,macro_recall,macro_f1,subset_accuracy
linear_svm,0.873418,0.850404,0.861757,0.815919,0.780169,0.797262,0.726543
logistic_regression,0.847826,0.828729,0.838169,0.768243,0.804252,0.779211,0.675364
naive_bayes,0.716547,0.634934,0.673276,0.612602,0.42141,0.457347,0.433016


Metrike po labelama za svaki classification model:


precision    recall        f1  support
model               label                                                  
logistic_regression cleanliness       0.933619  0.853229  0.891616    511.0
                    location          0.910990  0.844987  0.876748   1187.0
                    luxury            0.740558  0.787086  0.763113    573.0
                    family_friendly   0.487805  0.731707  0.585366     82.0
linear_svm          cleanliness       0.945607  0.884540  0.914055    511.0
                    location          0.902837  0.884583  0.893617   1187.0
                    luxury            0.785095  0.790576  0.787826    573.0
                    family_friendly   0.630137  0.560976  0.593548     82.0
naive_bayes         cleanliness       0.986395  0.283757  0.440729    511.0
                    location          0.673691  0.888795  0.766437   1187.0
                    luxury            0.790323  0.513089  0.622222    573.0
                    family_friendly   0.000000  0.000000  0.000000     82.0

In [11]:
regression_result = train_regression_models(df)
save_regression_result(regression_result)

regression_table = (
    pd.DataFrame(regression_result.all_metrics)
    .T
    .sort_values('mae', ascending=True)
)

print(f'Najbolji regression model: {regression_result.name}')
print('Metrike za sve regression modele:')
regression_table


Najbolji regression model: random_forest
Metrike za sve regression modele:


,mae,rmse,r2
random_forest,0.199312,0.253268,0.625959
ridge,0.201938,0.251351,0.631601
linear_regression,0.775706,1.221970,-7.707211


In [12]:
metrics_dir = PROJECT_ROOT / 'artifacts' / 'metrics'
classification_metrics = json.loads((metrics_dir / 'classification_metrics.json').read_text(encoding='utf-8'))
regression_metrics = json.loads((metrics_dir / 'regression_metrics.json').read_text(encoding='utf-8'))

classification_table = (
    pd.DataFrame(classification_metrics['models'])
    .T
    .drop(columns=['per_label'])
    .sort_values('micro_f1', ascending=False)
)
classification_per_label_table = pd.concat(
    {
        model_name: pd.DataFrame(metrics['per_label']).T
        for model_name, metrics in classification_metrics['models'].items()
    },
    names=['model', 'label'],
)
regression_table = (
    pd.DataFrame(regression_metrics['models'])
    .T
    .sort_values('mae', ascending=True)
)

print(f"Najbolji classification model: {classification_metrics['best_model']}")
print('Ukupne metrike za sve classification modele:')
display(classification_table)
print('Metrike po labelama za svaki classification model:')
display(classification_per_label_table)

print(f"Najbolji regression model: {regression_metrics['best_model']}")
print('Metrike za sve regression modele:')
regression_table


Najbolji classification model: linear_svm
Ukupne metrike za sve classification modele:


,micro_precision,micro_recall,micro_f1,macro_precision,macro_recall,macro_f1,subset_accuracy
linear_svm,0.873418,0.850404,0.861757,0.815919,0.780169,0.797262,0.726543
logistic_regression,0.847826,0.828729,0.838169,0.768243,0.804252,0.779211,0.675364
naive_bayes,0.716547,0.634934,0.673276,0.612602,0.42141,0.457347,0.433016


Metrike po labelama za svaki classification model:


precision    recall        f1  support
model               label                                                  
logistic_regression cleanliness       0.933619  0.853229  0.891616    511.0
                    location          0.910990  0.844987  0.876748   1187.0
                    luxury            0.740558  0.787086  0.763113    573.0
                    family_friendly   0.487805  0.731707  0.585366     82.0
linear_svm          cleanliness       0.945607  0.884540  0.914055    511.0
                    location          0.902837  0.884583  0.893617   1187.0
                    luxury            0.785095  0.790576  0.787826    573.0
                    family_friendly   0.630137  0.560976  0.593548     82.0
naive_bayes         cleanliness       0.986395  0.283757  0.440729    511.0
                    location          0.673691  0.888795  0.766437   1187.0
                    luxury            0.790323  0.513089  0.622222    573.0
                    family_friendly   0.000000  0.000000  0.000000     82.0

Najbolji regression model: random_forest
Metrike za sve regression modele:


,mae,rmse,r2
random_forest,0.199312,0.253268,0.625959
ridge,0.201938,0.251351,0.631601
linear_regression,0.775706,1.221970,-7.707211
